# 02 — Building an Indexing Pipeline That Retrieves Well

**Module 2.** Prepare organizational data so it can be indexed, vectorized,
and retrieved with high relevance.

**To build along:** set up your own environment first, using the README in
this repo — it takes about ten minutes. Run `setup/00_setup.ipynb` once
before this notebook. Everything here builds on that.

Run this notebook before `01_search_modes.ipynb`: Module 1 queries the
finished index this notebook produces, so there is no index for it to query
until Clip 3 below has run once.

Each clip section is self-contained after the **Setup** cells run once:
clients, config, and the two small helpers (`run_and_wait`, `show_hits`)
they all share. Re-run a clip's cells independently without re-running the
ones before it, as long as Setup has run in this kernel session.


## Setup — run once per kernel session

In [ ]:
import time

from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    AzureOpenAIEmbeddingSkill,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    CognitiveServicesAccountKey,
    FieldMapping,
    FieldMappingFunction,
    HnswAlgorithmConfiguration,
    IndexingParameters,
    IndexingParametersConfiguration,
    IndexProjectionMode,
    InputFieldMappingEntry,
    MergeSkill,
    OcrSkill,
    OutputFieldMappingEntry,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SearchIndexer,
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    SearchIndexerSkillset,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    SplitSkill,
    VectorSearch,
    VectorSearchProfile,
)
from azure.search.documents.models import VectorizableTextQuery

import sys
sys.path.insert(0, "..")

from config import load_config

config = load_config()
credential = DefaultAzureCredential()

index_client = SearchIndexClient(config.search_endpoint, credential)
indexer_client = SearchIndexerClient(config.search_endpoint, credential)
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

DATA_SOURCE_NAME = "state-driver-manuals-blob"
SKILLSET_NAME = "state-driver-manuals-skillset"
INDEXER_NAME = "state-driver-manuals-indexer"
INDEX_NAME = config.search_index_name

print("Search endpoint:", config.search_endpoint)
print("Index name:     ", INDEX_NAME)


In [ ]:
def reset_indexer_with_retry(retries=6, delay=5):
    """reset_indexer() rejects a reset while the service still considers a
    just-finished run 'in progress' server-side for a few seconds after
    last_result already reports success — retry through that window rather
    than trying to detect it."""
    for attempt in range(retries):
        try:
            indexer_client.reset_indexer(INDEXER_NAME)
            return
        except HttpResponseError as e:
            if attempt == retries - 1:
                raise
            print(f"(reset busy, retrying in {delay}s: {e.message})")
            time.sleep(delay)


def run_and_wait(reset=False, poll_seconds=5):
    """Run the indexer and block until it finishes. `reset=True` clears the
    indexer's change-tracking cache first — required whenever a skillset or
    index change should re-process documents it already saw.

    Right after run_indexer(), get_indexer_status() can still return the
    *previous* run's last_result for a moment — this waits for a result
    whose start_time actually changed before treating it as done. Note
    status.status ('running'/'error') reflects whether the indexer is
    operational, not whether it's executing right now — last_result.status
    ('inProgress'/'success'/...) is the one that reflects a live run."""
    if reset:
        reset_indexer_with_retry()
        previous_start = None
    else:
        prior = indexer_client.get_indexer_status(INDEXER_NAME).last_result
        previous_start = prior.start_time if prior else None
    indexer_client.run_indexer(INDEXER_NAME)
    print("Indexer running", end="", flush=True)
    while True:
        status = indexer_client.get_indexer_status(INDEXER_NAME)
        last = status.last_result
        is_new_result = last is not None and last.start_time != previous_start
        if is_new_result and last.status in ("success", "transientFailure", "persistentFailure"):
            print()
            print(f"Status: {last.status} — {last.item_count} item(s), {len(last.errors)} error(s)")
            for err in last.errors[:5]:
                print("  ERROR:", err.error_message)
            return last
        print(".", end="", flush=True)
        time.sleep(poll_seconds)


def build_index_projection(include_vector=False, include_state=False):
    """Rebuilt from scratch every time it's needed, rather than mutated on a
    skillset fetched from the service — `get_skillset()` returns
    `index_projection` as a plain dict, not the typed model, so appending to
    `.selectors[0].mappings` raises AttributeError. Simpler to always
    construct the full projection than to patch a dict/object mismatch."""
    mappings = [
        InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
        InputFieldMappingEntry(name="title", source="/document/title"),
    ]
    if include_vector:
        mappings.append(InputFieldMappingEntry(name="vector", source="/document/pages/*/vector"))
    if include_state:
        mappings.append(InputFieldMappingEntry(name="state", source="/document/state"))
    return SearchIndexerIndexProjection(
        selectors=[
            SearchIndexerIndexProjectionSelector(
                target_index_name=INDEX_NAME,
                parent_key_field_name="parent_id",
                source_context="/document/pages/*",
                mappings=mappings,
            ),
        ],
        parameters=SearchIndexerIndexProjectionsParameters(
            projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS,
        ),
    )


def show_hits(results, fields=("state", "title", "chunk"), snippet_len=180):
    for i, r in enumerate(results, start=1):
        line = f"{i}. score={r['@search.score']:.3f}"
        if r.get("@search.reranker_score") is not None:
            line += f"  reranker={r['@search.reranker_score']:.3f}"
        print(line)
        for field in fields:
            if field not in r:
                continue
            value = r[field]
            if field == "chunk" and isinstance(value, str):
                value = value.strip().replace("\n", " ")[:snippet_len] + "..."
            print(f"   {field}: {value}")
        print()


---
## Clip 1 (3 min) — Data Source, Skillset, Indexer, Index

Builds the four-part pipeline against the PDFs `00_setup.ipynb` uploaded to
Blob: OCR the image-only pages, merge that text back into the document,
split into ~2000-character page chunks, and project each chunk into its
own search document (not one giant document per PDF).

`ENABLE_OCR` is a toggle, not a permanent setting — flip it off, run once,
and query for a hazardous-load sign to get zero results; flip it on and
re-run to get hits. That contrast is the payoff for this clip — try it
yourself rather than taking it on trust.


In [ ]:
data_source = SearchIndexerDataSourceConnection(
    name=DATA_SOURCE_NAME,
    type="azureblob",
    # ResourceId= form + no key: the indexer authenticates as the search
    # service's own system-assigned managed identity (granted Storage Blob
    # Data Reader by infra/deploy.sh), not an account key.
    connection_string=(
        f"ResourceId=/subscriptions/{config.subscription_id}/resourceGroups/"
        f"{config.resource_group}/providers/Microsoft.Storage/storageAccounts/"
        f"{config.storage_account_name}/;"
    ),
    container=SearchIndexerDataContainer(name=config.storage_container_name),
)
indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{DATA_SOURCE_NAME}' ready.")


In [ ]:
# Index schema, v1: just enough to hold one chunk per search document.
# `chunk_id` is populated automatically by the index projection below — you
# don't map anything to it.
index_v1 = SearchIndex(
    name=INDEX_NAME,
    fields=[
        # SimpleField() silently drops analyzer_name (it's swallowed by the
        # helper's **kw), and Azure Search requires the keyword analyzer on
        # a key field used by an index projection — use SearchField directly.
        SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True,
                    analyzer_name="keyword", filterable=True),
        SimpleField(name="parent_id", type=SearchFieldDataType.String,
                    filterable=True),
        SearchField(name="title", type=SearchFieldDataType.String,
                    filterable=True, sortable=False),
        SearchField(name="chunk", type=SearchFieldDataType.String,
                    searchable=True),
    ],
)
index_client.create_or_update_index(index_v1)
print(f"Index '{INDEX_NAME}' created (v1: title + chunk, no vector field yet).")


In [ ]:
ENABLE_OCR = True  # flip to False, run once, then back to True to show the
                    # before/after in the portal per the clip note above.

skills = []
split_source = "/document/content"

if ENABLE_OCR:
    skills.append(OcrSkill(
        context="/document/normalized_images/*",
        default_language_code="en",
        should_detect_orientation=True,
        inputs=[InputFieldMappingEntry(name="image", source="/document/normalized_images/*")],
        outputs=[OutputFieldMappingEntry(name="text", target_name="text")],
    ))
    skills.append(MergeSkill(
        context="/document",
        insert_pre_tag=" ",
        insert_post_tag=" ",
        inputs=[
            InputFieldMappingEntry(name="text", source="/document/content"),
            InputFieldMappingEntry(name="itemsToInsert", source="/document/normalized_images/*/text"),
            InputFieldMappingEntry(name="offsets", source="/document/normalized_images/*/contentOffset"),
        ],
        outputs=[OutputFieldMappingEntry(name="mergedText", target_name="merged_text")],
    ))
    split_source = "/document/merged_text"

skills.append(SplitSkill(
    context="/document",
    text_split_mode="pages",
    maximum_page_length=2000,
    page_overlap_length=500,
    inputs=[InputFieldMappingEntry(name="text", source=split_source)],
    outputs=[OutputFieldMappingEntry(name="textItems", target_name="pages")],
))

index_projection = build_index_projection()

skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="OCR image-only pages, merge into content, split into page chunks",
    skills=skills,
    index_projection=index_projection,
)
if ENABLE_OCR:
    # Billing/quota attachment for the OCR skill. Key-based is the only
    # option for this specific attachment (unlike the embedding skill and
    # vectorizer below, which run keyless on the search service's managed
    # identity) — the free default caps OCR at 20 documents/day, which a
    # few practice runs will blow through.
    skillset.cognitive_services_account = CognitiveServicesAccountKey(key=config.ai_services_key)

indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{SKILLSET_NAME}' ready. OCR enabled: {ENABLE_OCR}")


In [ ]:
indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=DATA_SOURCE_NAME,
    target_index_name=INDEX_NAME,
    skillset_name=SKILLSET_NAME,
    parameters=IndexingParameters(
        configuration=IndexingParametersConfiguration(
            data_to_extract="contentAndMetadata",
            image_action="generateNormalizedImages" if ENABLE_OCR else "none",
            # The model defaults query_timeout to "00:05:00" even when you
            # never set it, and the service rejects that property on a blob
            # data source — explicit None is required to drop it.
            query_timeout=None,
        ),
    ),
    field_mappings=[
        FieldMapping(source_field_name="metadata_storage_name", target_field_name="title"),
    ],
)
indexer_client.create_or_update_indexer(indexer)
# reset=True: without it, an indexer with this name that has run before
# (e.g. a prior run, or toggling ENABLE_OCR and re-running this same
# cell) skips blobs its change-tracking already saw and reports 0 items —
# even if the target index is empty. That would silently break the
# ENABLE_OCR before/after demo below, so always force a full reprocess here.
result = run_and_wait(reset=True)


In [ ]:
results = search_client.search(search_text="FLAMMABLE radioactive placard", top=3, select=["title", "chunk"])
show_hits(results, fields=("title", "chunk"))
print(
    "Zero hits above with ENABLE_OCR = False (the placard wording lives inside "
    "the California p.42 sign-chart images, in no manual's extracted text). "
    "Portal check: Search service > Indexers > the run above, then open the "
    "index in Search Explorer and look at a chunk from the CA sign-chart page."
)


---
## Clip 2 (4 min) — Integrated Vectorization for Text and Image Content

Adds the embedding skill to the skillset and the matching vectorizer to the
index, so the service both embeds at indexing time and turns the query text
into a vector at search time — no client-side embedding call anywhere in
this section.


In [ ]:
# Vector search config: how to store/search vectors (HNSW), and how to turn
# query *text* into a vector automatically (the vectorizer).
vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="openai-vectorizer",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url=config.openai_endpoint,
        deployment_name=config.openai_embedding_deployment,
        model_name=config.openai_embedding_model,
    ),
    # No api_key: keyless, via the search service's managed identity
    # (granted Cognitive Services OpenAI User by infra/deploy.sh).
)

index_v2 = index_client.get_index(INDEX_NAME)
index_v2.fields.append(
    SearchField(
        name="vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        # stored=False (skip retrieving the raw vector, saving storage) is
        # only settable when a field is part of the original index — the
        # service rejects it on a field added via a later update, which is
        # what Clip 2 is doing here.
        vector_search_dimensions=1536,
        vector_search_profile_name="vector-profile",
    )
)
index_v2.vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[VectorSearchProfile(
        name="vector-profile",
        algorithm_configuration_name="hnsw-config",
        vectorizer_name="openai-vectorizer",
    )],
    vectorizers=[vectorizer],
)
index_client.create_or_update_index(index_v2)
print("Index updated: 'vector' field + vectorizer added.")


In [ ]:
embedding_skill = AzureOpenAIEmbeddingSkill(
    context="/document/pages/*",
    resource_url=config.openai_endpoint,
    deployment_name=config.openai_embedding_deployment,
    model_name=config.openai_embedding_model,
    dimensions=1536,
    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
    outputs=[OutputFieldMappingEntry(name="embedding", target_name="vector")],
)

skillset = indexer_client.get_skillset(SKILLSET_NAME)
skillset.skills = list(skillset.skills) + [embedding_skill]
skillset.index_projection = build_index_projection(include_vector=True)
indexer_client.create_or_update_skillset(skillset)
print("Skillset updated: embedding skill added, projection now maps 'vector' too.")

result = run_and_wait(reset=True)  # reset: force re-embedding of every chunk


In [ ]:
query = "How much space should I leave in front of me?"  # "following distance" in the manuals — the paraphrase never appears verbatim

results = search_client.search(
    search_text=None,
    vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=5, fields="vector")],
    top=3,
    select=["title", "chunk"],
)
show_hits(results, fields=("title", "chunk"))
print("Note: this notebook never called the embedding model directly — the vectorizer on the index did it, for both sides.")


Optional: drop a sixth state's manual into
its own folder in the `docs` container and re-run `run_and_wait(reset=True)`
above with no other code change — chunked, OCR'd, and embedded the same way.
The Azure AI Vision multimodal embedding skill (embedding images directly,
rather than OCR'ing them to text first) covers the same ground for
image-heavy content but is still in preview — worth knowing about,
not a demo.


---
## Clip 3 (3 min) — Filters and Semantic Ranking

Adds `state` as a filterable field derived straight from the blob folder —
one line in the indexer's field mappings, no custom skill — then shows a
filter changing the answer, and semantic ranking reordering results without
changing which documents came back.


In [ ]:
index_v3 = index_client.get_index(INDEX_NAME)
index_v3.fields.append(
    SearchField(name="state", type=SearchFieldDataType.String, filterable=True, facetable=True)
)
index_client.create_or_update_index(index_v3)

indexer = indexer_client.get_indexer(INDEXER_NAME)
indexer.field_mappings = list(indexer.field_mappings) + [
    FieldMapping(
        source_field_name="metadata_storage_path",
        target_field_name="state",
        # metadata_storage_path is the blob's full URL, e.g.
        # https://<account>.blob.core.windows.net/docs/new-jersey/<file>.pdf
        # — token 4 (0-indexed, split on '/') is the state folder. If your
        # storage account/container names change, re-count from a sample URL.
        mapping_function=FieldMappingFunction(name="extractTokenAtPosition", parameters={"delimiter": "/", "position": 4}),
    ),
]
indexer_client.create_or_update_indexer(indexer)

skillset = indexer_client.get_skillset(SKILLSET_NAME)
skillset.index_projection = build_index_projection(include_vector=True, include_state=True)
indexer_client.create_or_update_skillset(skillset)

result = run_and_wait(reset=True)


In [ ]:
query = "How close to a fire hydrant can I park?"  # CA/VA 15 ft, NJ 10 ft — a real, not cosmetic, split

print("-- unfiltered: states mixed together --")
show_hits(search_client.search(search_text=query, top=5, select=["state", "title", "chunk"]))

print("-- filter='state eq \'new-jersey\'' --")
# state values are the lowercase, hyphenated blob folder names (extractTokenAtPosition
# on metadata_storage_path), not the display name — 'new-jersey', not 'New Jersey'.
show_hits(search_client.search(search_text=query, filter="state eq 'new-jersey'", top=5, select=["state", "title", "chunk"]))


In [ ]:
control_query = "How far from a railroad crossing must I stop when signals are active?"  # ~100 ft everywhere — the control: not every question needs a filter

print("-- unfiltered control question: answer doesn't vary by state --")
show_hits(search_client.search(search_text=control_query, top=5, select=["state", "title", "chunk"]))


In [ ]:
index_v4 = index_client.get_index(INDEX_NAME)
index_v4.semantic_search = SemanticSearch(
    configurations=[
        SemanticConfiguration(
            name="semantic-config",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="title"),
                content_fields=[SemanticField(field_name="chunk")],
            ),
        )
    ],
    default_configuration_name="semantic-config",
)
index_client.create_or_update_index(index_v4)
print("Semantic configuration added — no re-run of the indexer needed, this only affects query-time ranking.")


In [ ]:
fire_hydrant_query = "How close to a fire hydrant can I park?"

results = search_client.search(
    search_text=fire_hydrant_query,
    query_type="semantic",
    semantic_configuration_name="semantic-config",
    top=5,
    select=["state", "title", "chunk"],
)
show_hits(results, fields=("state", "title", "chunk"))
print("Same candidate set as the unfiltered BM25 run above — compare the ordering and the reranker score column.")


---
## Done

`INDEX_NAME` now has chunked, OCR'd, embedded, state-filterable, and
semantically-ranked content from all five manuals. This is the finished
index Module 1's two demos (`01_search_modes.ipynb`) query.
